# Stable-Context: Brazil — Exploratory Analysis

Reproducible version of the key observations documented in `docs/analysis-findings/`.

Data: `data/gold/monthly_context.csv` and the underlying Gold datasets.

> Methodology note: observed data and derived metrics only. Signals are
> listed for investigation, not as conclusions.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
context = pd.read_csv(ROOT / "data/gold/monthly_context.csv", parse_dates=["month"])

print(f"Rows: {len(context)} | {context['month'].min().date()} -> {context['month'].max().date()}")
context.head(3)

## 1. Cross-source evolution (key months)

| month | receita (R$/m) | exchange (R$/m est.) | graph deposits (US$/m) |
|---|---|---|---|


In [ ]:
key = ["2023-01-01", "2024-01-01", "2025-01-01", "2025-07-01",
        "2026-01-01", "2026-06-01", "2026-07-01", "2026-09-01"]
sel = context[context["month"].isin(pd.to_datetime(key))].copy()
sel["receita_bi"] = sel["receita_stablecoin_brl"] / 1e9
sel["exchange_bi"] = sel["exchange_volume_brl_estimated"] / 1e9
sel["graph_bi"] = sel["deposits_usd"] / 1e9
sel[["month", "receita_bi", "exchange_bi", "graph_bi"]]

## 2. Exchange regime change (2026-07)

Observed in finding 03: ~10x step in July 2026, Foxbit-driven.


In [ ]:
exchange = pd.read_csv(ROOT / "data/gold/exchange_stablecoin_activity.csv",
    parse_dates=["date"])
exchange["month"] = exchange["date"].dt.to_period("M").dt.to_timestamp()
monthly = exchange.groupby("month").apply(
    lambda d: pd.Series({
        "brl_total": d["volume_brl"].sum(),
        "foxbit_share": d.loc[d["exchange"] == "Foxbit", "volume_brl"].sum()
                       / d["volume_brl"].sum() * 100,
    }), include_groups=False)
monthly_2026 = monthly.loc["2026-01":]
monthly_2026["brl_bi"] = monthly_2026["brl_total"] / 1e9
monthly_2026[["brl_bi", "foxbit_share"]].round(2)

## 3. USDC convergence (2026)

| source | USDC share |
|---|---|
| Receita (declared value) | 5% -> 29.3% (2026-06) |
| Exchanges (BRL notional) | 49.7% of 2026 YTD |
| The Graph deposits | USDT-led -> USDC-led from 2026-05 |


In [ ]:
receita = pd.read_csv(ROOT / "data/gold/receita_stablecoin_activity.csv", parse_dates=["month"])
st = receita[receita["stablecoin_flag"]].groupby("month")["total_value_brl"].sum()
usdc = receita[(receita["stablecoin_flag"]) & (receita["asset"] == "USDC")].groupby("month")["total_value_brl"].sum()
usdc_share = (usdc / st * 100).loc["2025-01":]
usdc_share.rename("usdc_share_pct").round(1)

## 4. Normalized monthly index

Index (first month = 100), as in `src/analysis/plot_monthly_context.py`.
Lines break where a source has no observation (Receita ends 2026-06).


In [ ]:
metrics = ["receita_stablecoin_brl", "exchange_volume_brl_estimated",
           "deposits_usd", "borrows_usd", "selic_meta", "usd_brl"]
idx = context[["month"] + metrics].copy()
for m in metrics:
    idx[m] = idx[m] / idx[m].iloc[0] * 100

plt.figure(figsize=(13, 7))
for m in metrics:
    plt.plot(idx["month"], idx[m], label=m)
plt.title("Stable-Context: Brazil — Monthly Context Index")
plt.xlabel("Month"); plt.ylabel("Index (first month = 100)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.show()

## Interpretation boundary

- Values above are observed or derived. Coincidences are signals.
- Context for investigation: `docs/context/brazil-context-events.md`.
- Full findings: `docs/analysis-findings/`.
